# **7주차: Computer Vision**

In [ ]:
!pip install torch
!pip install torchvision

### **데이터 불러오기 및 전처리**

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([transforms.ToTensor(),                                  # transform to tensor and normalize 0~255 to 0~1
                                transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])     # mean, std per channel for normalize -> normalize 0~1 to -1~1
train_dataset = torchvision.datasets.CIFAR10(root="./", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

### **CNN 모델 생성**

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# model definition
class MyModel(nn.Module):
    def __init__(self, num_labels):
        super(MyModel, self).__init__()
        self.conv1 = self.make_conv_block(3,16)        # output size = (16,16)
        self.conv2 = self.make_conv_block(16,16)       # output size = (8,8)
        self.avg_pool = nn.AvgPool2d(8)                # filter size = (8,8) -> global average pooling
        self.fc = nn.Linear(16,num_labels)

    def make_conv_block(self, in_c, out_c):
        return nn.Sequential(nn.Conv2d(in_channels=in_c,
                                       out_channels=out_c,
                                       kernel_size=3,
                                       stride=1,
                                       padding=1),
                             nn.BatchNorm2d(out_c),     # number of channels
                             nn.ReLU(),
                             nn.MaxPool2d(2))
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.avg_pool(x).view(x.size(0),-1)
        x = self.fc(x)
        return x

# model creation
model = MyModel(num_labels=10)

# display model structure
print(model)

### **모델 학습**

In [ ]:
# hyperparameter setting
epochs = 3
batch_size = 64
lr = 1e-4
weight_decay = 1e-2

# loss function, optimizer setting
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# dataset, dataloader creation
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# model training
for epoch in range(1,epochs+1):
    train_loss, train_acc, test_loss, test_acc = [], [], [], []
    model.train()                              # training mode of model
    for data, label in train_dataloader:
        pred = model(data)
        loss = criterion(pred, label)
        train_loss.append(loss)
        train_acc.append((pred.argmax(dim=-1)==label).sum() / len(label))

        optimizer.zero_grad()                  # remove accumulated gradient of parameters
        loss.backward()                        # gradient accumulation
        optimizer.step()                       # parameter update by accumulated gradient and optimizer

    model.eval()                               # evaluation mode of model
    for data, label in test_dataloader:
        with torch.no_grad():                  # turn off autograd (for calculation speed, memory efficiency)
            pred = model(data)
            loss = criterion(pred, label)
            test_loss.append(loss)
            test_acc.append((pred.argmax(dim=-1)==label).sum() / len(label))

    train_loss = torch.stack(train_loss).mean().item()
    train_acc = torch.stack(train_acc).mean().item()
    test_loss = torch.stack(test_loss).mean().item()
    test_acc = torch.stack(test_acc).mean().item()

    print(f"Epoch {epoch}/{epochs}")
    print(f"train loss: {round(train_loss, 4)}  train accuracy: {round(train_acc, 4)}")
    print(f"test loss: {round(test_loss, 4)}  test accuracy: {round(test_acc, 4)}\n")

### **사전 학습 모델 불러오기**

In [ ]:
import torchvision

In [ ]:
# load pretrained model
model = torchvision.models.resnet18(
    weights=torchvision.models.ResNet18_Weights.DEFAULT     # pretrained weights
)

# display model structure
print(model)

### **전이 학습**

In [ ]:
# change fc layer for our dataset (num labels = 10)
model.fc = nn.Linear(model.fc.in_features, 10)

# freeze all parameters except for classifier
for name, param in model.named_parameters():
    if "fc" not in name:
        param.requires_grad_(False)

# hyperparameter setting
epochs = 3
batch_size = 64 
lr = 1e-4
weight_decay = 1e-2

# loss function, optimizer setting
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# dataset, dataloader creation
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# model training
for epoch in range(1,epochs+1):
    train_loss, train_acc, test_loss, test_acc = [], [], [], []
    model.train()                              # training mode of model
    for data, label in train_dataloader:
        pred = model(data)
        loss = criterion(pred, label)
        train_loss.append(loss)
        train_acc.append((pred.argmax(dim=-1)==label).sum() / len(label))

        optimizer.zero_grad()                  # remove accumulated gradient of parameters
        loss.backward()                        # gradient accumulation
        optimizer.step()                       # parameter update by accumulated gradient and optimizer

    model.eval()                               # evaluation mode of model
    for data, label in test_dataloader:
        with torch.no_grad():                  # turn off autograd (for calculation speed, memory efficiency)
            pred = model(data)
            loss = criterion(pred, label)
            test_loss.append(loss)
            test_acc.append((pred.argmax(dim=-1)==label).sum() / len(label))

    train_loss = torch.stack(train_loss).mean().item()
    train_acc = torch.stack(train_acc).mean().item()
    test_loss = torch.stack(test_loss).mean().item()
    test_acc = torch.stack(test_acc).mean().item()

    print(f"Epoch {epoch}/{epochs}")
    print(f"train loss: {round(train_loss, 4)}  train accuracy: {round(train_acc, 4)}")
    print(f"test loss: {round(test_loss, 4)}  test accuracy: {round(test_acc, 4)}\n")